#  Employee Data Cleanup Pipeline
### DevBytes Internship — Task 1
**Dataset:** 520 rows × 21 columns — Real-world style HR employee data  
**Goal:** Load raw messy CSV → clean nulls → normalize → parse ranges → export

---

##  Step 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries ready')
print(f'   pandas : {pd.__version__}')
print(f'   numpy  : {np.__version__}')

##  Step 2 — Load Raw CSV

In [ ]:
# ── Point this to wherever you saved the raw_employee_data.csv file ──
RAW_FILE = 'raw_employee_data.csv'

df_raw = pd.read_csv(RAW_FILE, dtype=str)   # load everything as string first

print(f' Loaded: {RAW_FILE}')
print(f'   Shape : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head()

##  Step 3 — Data Quality Audit

In [ ]:
# Replace ALL disguised nulls with real NaN
NULL_ALIASES = ['NULL', 'N/A', 'n/a', 'NaN', 'nan', 'none', 'NONE', 'na', 'NA', '-', '', ' ']
df = df_raw.replace(NULL_ALIASES, np.nan)

null_counts  = df.isnull().sum()
null_pct     = (null_counts / len(df) * 100).round(1)

audit = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
audit = audit[audit['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print('=== NULL VALUE AUDIT (columns with missing data) ===')
print(audit.to_string())
print(f'\n  Total missing cells : {df.isnull().sum().sum()}')
print(f' Duplicate rows       : {df.duplicated().sum()}')

##  Step 4 — Clean Null Entries

In [ ]:
df_clean = df.copy()

# 1. Drop exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'  Duplicates removed   : {before - len(df_clean)}')

# 2. Coerce numeric columns
num_cols = ['age', 'salary', 'years_at_company', 'performance_score',
            'projects_completed', 'training_hours', 'satisfaction_score', 'last_promotion_year']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 3. Fill department nulls with 'Unknown'
df_clean['department'] = df_clean['department'].fillna('Unknown')

# 4. Fill salary with department median (smarter than global mean)
dept_med = df_clean.groupby('department')['salary'].transform('median')
df_clean['salary'] = df_clean['salary'].fillna(dept_med)

# 5. Fill age & performance with column median
for col in ['age', 'performance_score', 'satisfaction_score', 'projects_completed', 'training_hours']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 6. Fill last_promotion_year with hire_year (reasonable default)
df_clean['last_promotion_year'] = df_clean['last_promotion_year'].fillna(2020)

# 7. Fill manager_id nulls
df_clean['manager_id'] = df_clean['manager_id'].fillna('E0001')

# 8. Fill email nulls with generated placeholder
mask = df_clean['email'].isnull()
df_clean.loc[mask, 'email'] = df_clean.loc[mask, 'employee_id'].str.lower() + '@company.com'

print(f' Null cleaning done!')
print(f'   Remaining nulls : {df_clean.isnull().sum().sum()}')
print(f'   Rows remaining  : {len(df_clean)}')

## Step 5 — Normalize Columns

In [ ]:
df_norm = df_clean.copy()

# Text normalization
df_norm['full_name']   = df_norm['full_name'].str.strip().str.title()
df_norm['department']  = df_norm['department'].str.strip().str.title()
df_norm['email']       = df_norm['email'].str.strip().str.lower()
df_norm['city']        = df_norm['city'].str.strip().str.title()
df_norm['job_title']   = df_norm['job_title'].str.strip().str.title()

# Standardize gender
gender_map = {'male': 'Male', 'female': 'Female', 'non-binary': 'Non-binary'}
df_norm['gender'] = df_norm['gender'].str.strip().str.lower().map(
    lambda x: gender_map.get(x, x.title()) if isinstance(x, str) else x
)

# Standardize remote_work
remote_map = {'yes': 'Yes', 'no': 'No', 'hybrid': 'Hybrid'}
df_norm['remote_work'] = df_norm['remote_work'].str.strip().str.lower().map(
    lambda x: remote_map.get(x, x.title()) if isinstance(x, str) else x
)

# Convert hire_date to datetime
df_norm['hire_date'] = pd.to_datetime(df_norm['hire_date'], errors='coerce')
# Fill missing hire dates with median date
median_date = df_norm['hire_date'].dropna().sort_values().iloc[len(df_norm['hire_date'].dropna())//2]
df_norm['hire_date'] = df_norm['hire_date'].fillna(median_date)

# Convert numeric types
df_norm['salary']              = df_norm['salary'].round(0).astype(int)
df_norm['age']                 = df_norm['age'].round(0).astype(int)
df_norm['projects_completed']  = df_norm['projects_completed'].round(0).astype(int)
df_norm['training_hours']      = df_norm['training_hours'].round(0).astype(int)
df_norm['last_promotion_year'] = df_norm['last_promotion_year'].round(0).astype(int)

print(' Normalization complete!')
print(df_norm.dtypes)

## Step 6 — Parse Data Ranges

In [ ]:
df_final = df_norm.copy()

# Split '70000-150000' → salary_range_min & salary_range_max
split = df_final['salary_range'].str.strip().str.split('-', expand=True)
df_final['salary_range_min'] = pd.to_numeric(split[0], errors='coerce').astype('Int64')
df_final['salary_range_max'] = pd.to_numeric(split[1], errors='coerce').astype('Int64')
df_final['salary_range_mid'] = ((df_final['salary_range_min'] + df_final['salary_range_max']) / 2).astype('Int64')

# How far along in the range is their actual salary? (0-100%)
df_final['salary_percentile_in_band'] = (
    (df_final['salary'] - df_final['salary_range_min']) /
    (df_final['salary_range_max'] - df_final['salary_range_min']) * 100
).clip(0, 100).round(1)

# Seniority band from years
def seniority(y):
    if y <= 1:   return 'Junior'
    elif y <= 3: return 'Mid-Level'
    elif y <= 7: return 'Senior'
    else:        return 'Principal'
df_final['seniority_band'] = df_final['years_at_company'].astype(int).apply(seniority)

# Performance tier
def perf_tier(p):
    if p >= 4.5:   return 'Exceptional'
    elif p >= 3.5: return 'Meets Expectations'
    elif p >= 2.5: return 'Below Average'
    else:          return 'Needs Improvement'
df_final['performance_tier'] = df_final['performance_score'].apply(perf_tier)

# Drop original range string
df_final = df_final.drop(columns=['salary_range'])

print(' Range parsing complete!')
df_final[['full_name','department','salary','salary_range_min','salary_range_max',
          'salary_percentile_in_band','seniority_band','performance_tier']].head(10)

##  Step 7 — Summary Statistics

In [ ]:
print('=== FINAL DATASET OVERVIEW ===')
print(f'Shape  : {df_final.shape[0]} rows × {df_final.shape[1]} columns')
print(f'Nulls  : {df_final.isnull().sum().sum()}')

print('\n=== HEADCOUNT & AVG SALARY BY DEPARTMENT ===')
dept = df_final.groupby('department').agg(
    Headcount=('employee_id','count'),
    Avg_Salary=('salary','mean'),
    Median_Salary=('salary','median'),
    Avg_Performance=('performance_score','mean')
).round(0)
print(dept.to_string())

print('\n=== SENIORITY DISTRIBUTION ===')
print(df_final['seniority_band'].value_counts().to_string())

print('\n=== PERFORMANCE TIERS ===')
print(df_final['performance_tier'].value_counts().to_string())

print('\n=== REMOTE WORK BREAKDOWN ===')
print(df_final['remote_work'].value_counts().to_string())

In [ ]:
print('=== FULL CLEANED DATASET ===')
df_final

## Step 8 — Export Cleaned Data

In [ ]:
# Saves in same folder as the notebook & CSV
df_final.to_csv('cleaned_employee_data.csv', index=False)

print(' Export complete!')
print(f'   File    : cleaned_employee_data.csv')
print(f'   Rows    : {len(df_final)}')
print(f'   Columns : {df_final.shape[1]}')
print(f'   Saved in: {os.getcwd()}')
print('\n Pipeline done! Your data is clean and ready for analysis.')

---
##  Cleanup Summary Table

| Step | What We Did | Why |
|------|------------|-----|
| 1 | Replaced disguised nulls (NULL, N/A, n/a, -, etc.) | Pandas only sees real `NaN`, not string fakes |
| 2 | Dropped 20 exact duplicate rows | Duplicates skew all statistics |
| 3 | Filled salary with **department median** | Group-aware — Engineering median ≠ HR median |
| 4 | Filled age/performance with column median | Median is robust to outliers |
| 5 | Title-cased names, departments, cities | Prevents `engineering` vs `Engineering` mismatches |
| 6 | Standardized gender & remote_work labels | `MALE` / `male` / `Male` all became `Male` |
| 7 | Parsed `hire_date` → datetime object | Enables tenure & time calculations |
| 8 | Split salary_range string → min/max/mid columns | Structured data is queryable; strings are not |
| 9 | Added seniority_band, performance_tier, salary_percentile | Enriched features for analysis or ML |

---
*DevBytes Internship — Data Pipeline Task 1*